In [9]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests

from concurrent.futures import ThreadPoolExecutor, as_completed
import time, zipfile, io

PROJECT_ROOT = Path("/home/py/groundwater")
TABLE_DIR = PROJECT_ROOT / "data" / "gown" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

BASE = "https://groundwatermonitoring.alberta.ca"
LAYERS = {"active": 10, "inactive": 20}

In [2]:
raw = {}
for category, layer in LAYERS.items():
    r = requests.get(f"{BASE}/data/internet/layers/{layer}/index.json", timeout=30)
    r.raise_for_status()
    raw[category] = r.json()
    (TABLE_DIR / f"layer{layer}_{category}.json").write_text(json.dumps(raw[category]))
    print(category, len(raw[category]))

active 310
inactive 854


## Metadata

In [45]:
def station_url(station_id: str, station_name: str, category: str) -> str:
    return f"{BASE}/#/overview/{category}wells/station/{station_id}/{quote(station_name)}?mode=table"

FIELDS = {
    "station_name": "station_name",
    "station_no":   "station_no",
    "gown":         "groundwater.StationCode_S",
    "station_id":   "station_id",
    "latitude":     "station_latitude",
    "longitude":    "station_longitude",
    "depth":        "groundwater.Depth_N",
    "water_quality": "groundwater.WaterQualityType_K",
    "data_type":    "groundwater.WebGWDataType",
    "catchment":    "catchment_name",
    "gw_type":      "groundwater.WISKIStatTemplate_K",
}

df_stations = pd.DataFrame([
    {"category": category,
     **{out: r.get(src) for out, src in FIELDS.items()},
     "url": station_url(r["station_id"], r["station_name"], category)}
    for category, records in raw.items()
    for r in records
])

df_stations.to_csv(TABLE_DIR / "stations.csv", index=False)

In [10]:
META_DIR = PROJECT_ROOT / "data" / "gown" / "meta"
META_DIR.mkdir(parents=True, exist_ok=True)

def fetch_meta(station_no: str, retries: int = 3):
    dest = META_DIR / f"{station_no}.json"
    if dest.exists():
        return station_no, "cached"
    for attempt in range(retries):
        try:
            r = requests.get(f"{BASE}/data/internet/stations/AB/{station_no}/index.json", timeout=30)
            if r.status_code == 404:
                return station_no, "404"
            r.raise_for_status()
            dest.write_text(r.text)
            return station_no, "ok"
        except Exception as e:
            if attempt == retries - 1:
                return station_no, f"fail: {type(e).__name__}"
            time.sleep(2 ** attempt)

station_nos = df_stations["station_no"].tolist()
with ThreadPoolExecutor(max_workers=6) as pool:
    out = list(pool.map(fetch_meta, station_nos))

pd.Series(dict(out)).value_counts()

ok        1154
404          6
cached       2
Name: count, dtype: int64

In [11]:
records = []
for sn in station_nos:
    p = META_DIR / f"{sn}.json"
    if not p.exists():
        continue
    d = json.loads(p.read_text())
    if isinstance(d, list):
        d = d[0] if len(d) == 1 else {"_list": d}
    records.append({"station_no": sn, **d})

df_meta = pd.json_normalize(records)
print(df_meta.shape)
list(df_meta.columns)

(1158, 36)


['station_no',
 'station_id',
 'station_name',
 'site_no',
 'site_name',
 'station_latitude',
 'station_longitude',
 'object_type',
 'catchment_name',
 'station_elevation',
 'CATCHMENT_SIZE',
 'WTO_OBJECT',
 'GAUGE_DATUM',
 'GWREF_DATUM',
 'DIST_TO_CONFL',
 'groundwater.WISKIStatTemplate_K',
 'groundwater.StationCode_S',
 'groundwater.GICWellID_S',
 'general.DrainageBasin_S',
 'groundwater.Depth_N',
 'general.LSD_S',
 'general.Section_S',
 'general.Township_S',
 'general.Range_S',
 'general.Meridian_S',
 'groundwater.Aquifer_S',
 'groundwater.Lithology_S',
 'groundwater.AquiferType_K',
 'groundwater.1stWQSampleDate_D',
 'groundwater.DrillDate_D',
 'groundwater.Production_S',
 'groundwater.DataCollection_K',
 'station_diary_status',
 'groundwater.WebGWDataType',
 'creationDateInMillis',
 '_links']

In [23]:
import re

DIARY_RE = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}):\s*(.+)$")

def parse_diary(s) -> list[tuple[pd.Timestamp, str]]:
    if not isinstance(s, str) or not s.strip():
        return []
    out = []
    for entry in s.split("<br>"):
        m = DIARY_RE.match(entry.strip())
        if m:
            out.append((pd.Timestamp(m.group(1)), m.group(2).strip()))
    return sorted(out)

df_meta["diary"] = df_meta["station_diary_status"].apply(parse_diary)
df_meta["last_status"] = df_meta["diary"].apply(lambda d: d[-1][1] if d else None)
df_meta["last_status_date"] = df_meta["diary"].apply(lambda d: d[-1][0] if d else pd.NaT)

df_meta["last_status"].value_counts(dropna=False)

last_status
Reclaimed                327
Inactive                 281
Active                   252
Active WL & WQ           103
Inactive WQ only          94
Asset Removed             35
Transferred               19
Inactive Project          15
Active WL only            12
Destroyed                 10
Metadata Only              4
Active WL Inactive WQ      3
Damaged                    1
Active WQ Inactive WL      1
Pending                    1
Name: count, dtype: int64

In [44]:
df_meta.to_csv(TABLE_DIR / "stations_meta.csv", index=False)

## Download RAW Data

In [37]:
RAW_DIR = PROJECT_ROOT / "data" / "gown" / "raw"
for cat in LAYERS:
    (RAW_DIR / cat).mkdir(parents=True, exist_ok=True)

def zip_url(station_no: str) -> str:
    return f"{BASE}/data/internet/stations/AB/{station_no}/WL/Waterlevel_complete.zip"

# GOWN # as filename, zero-padded so lexical order == numeric order
df_stations["fname"] = df_stations["gown"].astype(str).str.zfill(4)
assert not df_stations.duplicated(["category", "fname"]).any(), "GOWN # collision"

In [38]:
def fetch_one(row, retries: int = 3) -> tuple[str, str]:
    dest = RAW_DIR / row.category / f"{row.fname}.zip"
    if dest.exists() and dest.stat().st_size > 0:
        return row.fname, "skipped"

    tmp = dest.with_suffix(".part")
    for attempt in range(retries):
        try:
            with requests.get(zip_url(row.station_no), stream=True, timeout=60) as r:
                if r.status_code == 404:
                    return row.fname, "404"
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 16):
                        f.write(chunk)
            tmp.rename(dest)          # atomic: dest only exists once complete
            return row.fname, "ok"
        except Exception as e:
            if attempt == retries - 1:
                tmp.unlink(missing_ok=True)
                return row.fname, f"fail: {type(e).__name__}"
            time.sleep(2 ** attempt)  # 1s, 2s, 4s backoff

In [39]:
results = {}
t0 = time.time()
rows = list(df_stations.itertuples())

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(fetch_one, r): r for r in rows}
    for i, fut in enumerate(as_completed(futures), 1):
        name, status = fut.result()
        results[name] = status
        if i % 25 == 0:
            rate = i / (time.time() - t0)
            print(f"\r{i}/{len(rows)} | {rate:.1f}/s | ETA {(len(rows)-i)/rate/60:.1f} min", end="")

pd.Series(results).value_counts()

1150/1163 | 36.9/s | ETA 0.0 min

404    666
ok     496
Name: count, dtype: int64

In [40]:
def read_ts(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as z:
        name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
        with z.open(name) as f:
            df = pd.read_csv(
                f, sep=";", comment="#",
                names=["timestamp", "value", "absolute_value"],
                dtype={"value": "float32", "absolute_value": "float32"},
            )
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="ISO8601")
    return df.set_index("timestamp")

In [41]:
PARQUET_DIR = PROJECT_ROOT / "data" / "gown" / "parquet"
for cat in LAYERS:
    (PARQUET_DIR / cat).mkdir(parents=True, exist_ok=True)

for zp in sorted(RAW_DIR.rglob("*.zip")):
    out = PARQUET_DIR / zp.parent.name / f"{zp.stem}.parquet"
    if out.exists():
        continue
    read_ts(zp).to_parquet(out, compression="zstd")